In [2]:
import pandas as pd

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

In [3]:
print("Training dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

print("\nFirst 5 training rows:")
display(train_df.head())

print("\nFirst 5 test rows:")
display(test_df.head())

Training dataset shape: (20000, 3)
Test dataset shape: (6999, 2)

Training columns:
['id', 'text', 'label']

Test columns:
['id', 'text']

First 5 training rows:


,id,text,label
0,71ec6000-1f20-4850-a8f7-140bc6ad640d,Instance-level video segmentation requires a s...,1
1,9a494ddf-43c6-4c54-8927-f2343054fef2,Samples of high-redshift galaxies are easy to ...,0
2,bc1dc7e4-d773-4c52-bcc1-ed08ba822f92,"Ashley Sibery, 39, persuaded Sital Sibery to t...",1
3,c9f43db6-21b9-4ca6-9830-ce06ff47b950,On-shell methods offer an alternative definiti...,0
4,3c0ef5fb-5311-4185-b773-ea48e7e5bd51,The ocean goes through two tide cycles in a da...,1



First 5 test rows:


,id,text
0,59218,The paper presents an analytical derivation of...
1,37110,Summary\nThe paper reviews the potential cardi...
2,23200,This paper presents a study of different simul...
3,e3357348-166e-4847-a06d-158b7cd83aa5,Video conferencing has become very popular ove...
4,61615,Alternative splicing (AS) definitely contribut...


In [4]:
ID_COLUMN = "id"
LABEL_COLUMN = "label"
FEATURE_COLUMNS = ["text"]

print("ID column:", ID_COLUMN)
print("Label column:", LABEL_COLUMN)
print("Feature columns:", FEATURE_COLUMNS)
print("Number of features:", len(FEATURE_COLUMNS))

ID column: id
Label column: label
Feature columns: ['text']
Number of features: 1


In [5]:
print("Missing values in training data:")
display(train_df.isnull().sum())

print("Missing values in test data:")
display(test_df.isnull().sum())

Missing values in training data:


id       0
text     0
label    0
dtype: int64

Missing values in test data:


id      0
text    0
dtype: int64

In [7]:
class_distribution = train_df[LABEL_COLUMN].value_counts().sort_index()

class_distribution_table = pd.DataFrame({
    "Label": class_distribution.index,
    "Count": class_distribution.values,
    "Percentage": (class_distribution.values / len(train_df) * 100).round(2)
})

display(class_distribution_table)

,Label,Count,Percentage
0,0,7496,37.48
1,1,12504,62.52


In [8]:
dataset_summary = pd.DataFrame({
    "Dataset": ["Training", "Test"],
    "Rows": [train_df.shape[0], test_df.shape[0]],
    "Columns": [train_df.shape[1], test_df.shape[1]],
    "Number of features": [
        len(FEATURE_COLUMNS),
        len(FEATURE_COLUMNS)
    ],
    "Total missing values": [
        train_df.isnull().sum().sum(),
        test_df.isnull().sum().sum()
    ]
})

display(dataset_summary)

,Dataset,Rows,Columns,Number of features,Total missing values
0,Training,20000,3,1,0
1,Test,6999,2,1,0


In [9]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
VALIDATION_SIZE = 0.20

# Split the row indices instead of splitting the data directly
train_indices, validation_indices = train_test_split(
    train_df.index,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_SEED,
    stratify=train_df[LABEL_COLUMN]
)

print("Number of training rows:", len(train_indices))
print("Number of validation rows:", len(validation_indices))

Number of training rows: 16000
Number of validation rows: 4000


In [10]:
X_train = train_df.loc[train_indices, "text"]
X_validation = train_df.loc[validation_indices, "text"]

y_train = train_df.loc[train_indices, LABEL_COLUMN]
y_validation = train_df.loc[validation_indices, LABEL_COLUMN]

print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("y_train shape:", y_train.shape)
print("y_validation shape:", y_validation.shape)

X_train shape: (16000,)
X_validation shape: (4000,)
y_train shape: (16000,)
y_validation shape: (4000,)


In [11]:
split_distribution = pd.DataFrame({
    "Full dataset": train_df[LABEL_COLUMN].value_counts(normalize=True).sort_index(),
    "Training split": y_train.value_counts(normalize=True).sort_index(),
    "Validation split": y_validation.value_counts(normalize=True).sort_index()
}).multiply(100).round(2)

display(split_distribution)

,Full dataset,Training split,Validation split
label,,,
0,37.48,37.48,37.48
1,62.52,62.52,62.52


In [12]:
from pathlib import Path

# Mark every row as training first
split_labels = pd.Series(
    "train",
    index=train_df.index,
    name="split"
)

# Change the selected validation rows to validation
split_labels.loc[validation_indices] = "validation"

shared_split_df = pd.DataFrame({
    "row_index": train_df.index,
    "id": train_df[ID_COLUMN],
    "split": split_labels
})

# Create the splits folder if it does not exist
Path("../data/splits").mkdir(parents=True, exist_ok=True)

shared_split_df.to_csv(
    "../data/splits/shared_validation_split.csv",
    index=False
)

display(shared_split_df.head())

print("Shared split saved successfully.")

,row_index,id,split
0,0,71ec6000-1f20-4850-a8f7-140bc6ad640d,train
1,1,9a494ddf-43c6-4c54-8927-f2343054fef2,train
2,2,bc1dc7e4-d773-4c52-bcc1-ed08ba822f92,train
3,3,c9f43db6-21b9-4ca6-9830-ce06ff47b950,train
4,4,3c0ef5fb-5311-4185-b773-ea48e7e5bd51,validation


Shared split saved successfully.


In [13]:
assert len(train_indices) + len(validation_indices) == len(train_df)
assert set(train_indices).isdisjoint(set(validation_indices))

print("Split verified: every row belongs to exactly one split.")

Split verified: every row belongs to exactly one split.


In [14]:
import sys
from pathlib import Path

# Find the main project folder
PROJECT_ROOT = Path.cwd().parent

# Allow the notebook to import files from the project folder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation import calculate_macro_f1

print("Macro F1 function imported successfully.")

Macro F1 function imported successfully.


In [15]:
example_actual = [0, 0, 1, 1]
example_predictions = [0, 0, 1, 1]

example_score = calculate_macro_f1(
    example_actual,
    example_predictions
)

print("Example Macro F1:", example_score)

Example Macro F1: 1.0


In [16]:
from src.submission import create_submission

print("Submission function imported successfully.")

Submission function imported successfully.


In [17]:
dummy_predictions = [0, 1, 0, 1, 0]

format_check_path = (
    PROJECT_ROOT / "submissions" / "format_check.csv"
)

format_check_df = create_submission(
    test_ids=test_df[ID_COLUMN].head(5),
    predictions=dummy_predictions,
    output_path=format_check_path,
    id_column=ID_COLUMN,
    label_column=LABEL_COLUMN
)

display(format_check_df)

print("Format-check file saved to:")
print(format_check_path)

,id,label
0,59218,0
1,37110,1
2,23200,0
3,e3357348-166e-4847-a06d-158b7cd83aa5,1
4,61615,0


Format-check file saved to:
c:\Users\akshi\Downloads\TheHomiesML-1\submissions\format_check.csv


In [18]:
saved_format_check = pd.read_csv(
    format_check_path,
    dtype={ID_COLUMN: "string"}
)

expected_ids = (
    test_df[ID_COLUMN]
    .head(5)
    .astype("string")
    .reset_index(drop=True)
)

assert saved_format_check.columns.tolist() == [
    ID_COLUMN,
    LABEL_COLUMN
]

assert saved_format_check[ID_COLUMN].tolist() == expected_ids.tolist()

assert len(saved_format_check) == 5

print("Submission format verified successfully.")
print("The test IDs were preserved in their original order.")

Submission format verified successfully.
The test IDs were preserved in their original order.


In [19]:
dummy_predictions = [0, 1, 0, 1, 0]

format_check_path = (
    PROJECT_ROOT / "submissions" / "format_check.csv"
)

format_check_df = create_submission(
    test_ids=test_df[ID_COLUMN].head(5),
    predictions=dummy_predictions,
    output_path=format_check_path,
    id_column=ID_COLUMN,
    label_column=LABEL_COLUMN
)

display(format_check_df)

print("Format-check file saved to:")
print(format_check_path)

,id,label
0,59218,0
1,37110,1
2,23200,0
3,e3357348-166e-4847-a06d-158b7cd83aa5,1
4,61615,0


Format-check file saved to:
c:\Users\akshi\Downloads\TheHomiesML-1\submissions\format_check.csv


In [20]:
saved_format_check = pd.read_csv(
    format_check_path,
    dtype={ID_COLUMN: "string"}
)

expected_ids = (
    test_df[ID_COLUMN]
    .head(5)
    .astype("string")
    .reset_index(drop=True)
)

assert saved_format_check.columns.tolist() == [
    ID_COLUMN,
    LABEL_COLUMN
]

assert saved_format_check[ID_COLUMN].tolist() == expected_ids.tolist()

assert len(saved_format_check) == 5

print("Submission format verified successfully.")
print("The test IDs were preserved in their original order.")

Submission format verified successfully.
The test IDs were preserved in their original order.


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

In [22]:
baseline_svm = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "svm",
        LinearSVC(
            C=1.0,
            class_weight=None,
            random_state=RANDOM_SEED
        )
    )
])

print("Baseline Linear SVM created.")

Baseline Linear SVM created.


In [23]:
baseline_svm.fit(X_train, y_train)

print("Baseline Linear SVM trained successfully.")

Baseline Linear SVM trained successfully.


In [24]:
baseline_validation_predictions = baseline_svm.predict(
    X_validation
)

print(
    "Number of validation predictions:",
    len(baseline_validation_predictions)
)

print(
    "First 10 predictions:",
    baseline_validation_predictions[:10]
)

Number of validation predictions: 4000
First 10 predictions: [1 1 0 1 0 1 0 1 1 1]


In [25]:
baseline_macro_f1 = calculate_macro_f1(
    y_validation,
    baseline_validation_predictions
)

print(
    f"Baseline validation Macro F1: "
    f"{baseline_macro_f1:.6f}"
)

Baseline validation Macro F1: 0.741272


In [26]:
baseline_result = pd.DataFrame([
    {
        "Model": "Linear SVM Baseline",
        "C": 1.0,
        "Class Weight": "None",
        "Validation Macro F1": baseline_macro_f1
    }
])

display(baseline_result)

,Model,C,Class Weight,Validation Macro F1
0,Linear SVM Baseline,1.0,None,0.741272


In [27]:
tuning_tfidf = TfidfVectorizer()

# Learn the vocabulary only from the training text
X_train_tfidf = tuning_tfidf.fit_transform(X_train)

# Use the same learned vocabulary on the validation text
X_validation_tfidf = tuning_tfidf.transform(X_validation)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_validation_tfidf.shape)

Training TF-IDF shape: (16000, 82595)
Validation TF-IDF shape: (4000, 82595)


In [28]:
C_VALUES = [0.01, 0.1, 0.5, 1, 2, 5, 10]

svm_c_results = []

for c_value in C_VALUES:
    print(f"Testing C = {c_value}")

    svm_model = LinearSVC(
        C = c_value,
        class_weight = None,
        random_state = RANDOM_SEED
    )

    # Train using the training split
    svm_model.fit(X_train_tfidf, y_train)

    # Predict the validation split
    validation_predictions = svm_model.predict(
        X_validation_tfidf
    )

    # Calculate Macro F1
    macro_f1 = calculate_macro_f1(
        y_validation,
        validation_predictions
    )

    svm_c_results.append({
        "C": c_value,
        "Class Weight": "None",
        "Validation Macro F1": macro_f1
    })

    print(f"Macro F1: {macro_f1:.6f}\n")

Testing C = 0.01
Macro F1: 0.517050

Testing C = 0.1
Macro F1: 0.728306

Testing C = 0.5
Macro F1: 0.746982

Testing C = 1
Macro F1: 0.741272

Testing C = 2
Macro F1: 0.735353

Testing C = 5
Macro F1: 0.728043

Testing C = 10
Macro F1: 0.719784



In [29]:
svm_c_results_df = pd.DataFrame(svm_c_results)

display(
    svm_c_results_df.style.format({
        "C": "{:g}",
        "Validation Macro F1": "{:.6f}"
    })
)

,C,Class Weight,Validation Macro F1
0,0.01,None,0.517050
1,0.1,None,0.728306
2,0.5,None,0.746982
3,1,None,0.741272
4,2,None,0.735353
5,5,None,0.728043
6,10,None,0.719784


In [30]:
svm_c_results_ranked = (
    svm_c_results_df
    .sort_values(
        by="Validation Macro F1",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    svm_c_results_ranked.style.format({
        "C": "{:g}",
        "Validation Macro F1": "{:.6f}"
    })
)

,C,Class Weight,Validation Macro F1
0,0.5,None,0.746982
1,1,None,0.741272
2,2,None,0.735353
3,0.1,None,0.728306
4,5,None,0.728043
5,10,None,0.719784
6,0.01,None,0.517050


In [31]:
best_three_c_values = (
    svm_c_results_ranked
    .head(3)["C"]
    .tolist()
)

print("Best three C values:", best_three_c_values)

Best three C values: [0.5, 1.0, 2.0]


In [34]:
CLASS_WEIGHT_OPTIONS = [None, "balanced"]

svm_class_weight_results = []

for c_value in best_three_c_values:
    for class_weight_value in CLASS_WEIGHT_OPTIONS:

        print(
            f"Testing C = {c_value}, "
            f"class_weight = {class_weight_value}"
        )

        svm_model = LinearSVC(
            C=c_value,
            class_weight=class_weight_value,
            random_state=RANDOM_SEED
        )

        # Train the model
        svm_model.fit(
            X_train_tfidf,
            y_train
        )

        # Predict the validation labels
        validation_predictions = svm_model.predict(
            X_validation_tfidf
        )

        # Calculate Macro F1
        macro_f1 = calculate_macro_f1(
            y_validation,
            validation_predictions
        )

        svm_class_weight_results.append({
            "C": c_value,
            "Class Weight": (
                "None"
                if class_weight_value is None
                else class_weight_value
            ),
            "Validation Macro F1": macro_f1
        })

        print(f"Macro F1: {macro_f1:.6f}\n")

Testing C = 0.5, class_weight = None
Macro F1: 0.746982

Testing C = 0.5, class_weight = balanced
Macro F1: 0.754983

Testing C = 1.0, class_weight = None
Macro F1: 0.741272

Testing C = 1.0, class_weight = balanced
Macro F1: 0.750906

Testing C = 2.0, class_weight = None
Macro F1: 0.735353

Testing C = 2.0, class_weight = balanced
Macro F1: 0.739826



In [35]:
svm_class_weight_results_df = pd.DataFrame(
    svm_class_weight_results
)

display(
    svm_class_weight_results_df.style.format({
        "C": "{:g}",
        "Validation Macro F1": "{:.6f}"
    })
)

,C,Class Weight,Validation Macro F1
0,0.5,None,0.746982
1,0.5,balanced,0.754983
2,1,None,0.741272
3,1,balanced,0.750906
4,2,None,0.735353
5,2,balanced,0.739826


In [36]:
svm_class_weight_results_ranked = (
    svm_class_weight_results_df
    .sort_values(
        by="Validation Macro F1",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    svm_class_weight_results_ranked.style.format({
        "C": "{:g}",
        "Validation Macro F1": "{:.6f}"
    })
)

,C,Class Weight,Validation Macro F1
0,0.5,balanced,0.754983
1,1,balanced,0.750906
2,0.5,None,0.746982
3,1,None,0.741272
4,2,balanced,0.739826
5,2,None,0.735353


In [37]:
all_svm_results = pd.concat(
    [
        svm_c_results_df,
        svm_class_weight_results_df
    ],
    ignore_index=True
)

all_svm_results = all_svm_results.drop_duplicates(
    subset=["C", "Class Weight"],
    keep="first"
)

print("Number of unique configurations:", len(all_svm_results))

display(
    all_svm_results.style.format({
        "C": "{:g}",
        "Validation Macro F1": "{:.6f}"
    })
)

Number of unique configurations: 10


,C,Class Weight,Validation Macro F1
0,0.01,None,0.517050
1,0.1,None,0.728306
2,0.5,None,0.746982
3,1,None,0.741272
4,2,None,0.735353
5,5,None,0.728043
6,10,None,0.719784
8,0.5,balanced,0.754983
10,1,balanced,0.750906
12,2,balanced,0.739826


In [38]:
all_svm_results_ranked = (
    all_svm_results
    .sort_values(
        by="Validation Macro F1",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    all_svm_results_ranked.style.format({
        "C": "{:g}",
        "Validation Macro F1": "{:.6f}"
    })
)

,C,Class Weight,Validation Macro F1
0,0.5,balanced,0.754983
1,1,balanced,0.750906
2,0.5,None,0.746982
3,1,None,0.741272
4,2,balanced,0.739826
5,2,None,0.735353
6,0.1,None,0.728306
7,5,None,0.728043
8,10,None,0.719784
9,0.01,None,0.517050


In [40]:
best_svm_result = all_svm_results_ranked.iloc[0]

BEST_C = float(best_svm_result["C"])

BEST_CLASS_WEIGHT = best_svm_result["Class Weight"]

if BEST_CLASS_WEIGHT == "None":
    BEST_CLASS_WEIGHT = None

BEST_VALIDATION_MACRO_F1 = float(
    best_svm_result["Validation Macro F1"]
)

print("Best SVM configuration")
print("C:", BEST_C)
print("Class weight:", BEST_CLASS_WEIGHT)
print(
    f"Validation Macro F1: "
    f"{BEST_VALIDATION_MACRO_F1:.6f}"
)

Best SVM configuration
C: 0.5
Class weight: balanced
Validation Macro F1: 0.754983


In [41]:
best_svm_configuration = pd.DataFrame([
    {
        "Model": "Linear SVM",
        "C": BEST_C,
        "Class Weight": (
            "None"
            if BEST_CLASS_WEIGHT is None
            else BEST_CLASS_WEIGHT
        ),
        "Validation Macro F1": BEST_VALIDATION_MACRO_F1
    }
])

display(
    best_svm_configuration.style.format({
        "C": "{:g}",
        "Validation Macro F1": "{:.6f}"
    })
)

,Model,C,Class Weight,Validation Macro F1
0,Linear SVM,0.5,balanced,0.754983


In [43]:
final_svm = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "svm",
        LinearSVC(
            C=BEST_C,
            class_weight=BEST_CLASS_WEIGHT,
            random_state=RANDOM_SEED
        )
    )
])

print("Final SVM pipeline created")
print("Selected C:", BEST_C)
print("Selected class weight:", BEST_CLASS_WEIGHT)

Final SVM pipeline created
Selected C: 0.5
Selected class weight: balanced


In [44]:
X_full_train = train_df["text"]
y_full_train = train_df[LABEL_COLUMN]

print("Number of full training samples:", len(X_full_train))
print("Number of full training labels:", len(y_full_train))

Number of full training samples: 20000
Number of full training labels: 20000


In [ ]:
final_svm.fit(X_full_train, y_full_train)

print("Final Linear SVM trained successfully.")

Final Linear SVM trained successfully.


In [47]:
print("Final model details")
print("Training samples:", len(X_full_train))
print("C:", final_svm.named_steps["svm"].C)
print(
    "Class weight:",
    final_svm.named_steps["svm"].class_weight
)

Final model details
Training samples: 20000
C: 0.5
Class weight: balanced


In [48]:
X_test = test_df["text"]

print("Number of test samples:", len(X_test))

Number of test samples: 6999


In [49]:
svm_test_predictions = final_svm.predict(X_test)

print("Number of predictions:", len(svm_test_predictions))
print("First 10 predictions:", svm_test_predictions[:10])

Number of predictions: 6999
First 10 predictions: [1 1 0 1 1 1 1 1 1 0]


In [50]:
prediction_distribution = pd.Series(
    svm_test_predictions
).value_counts().sort_index()

prediction_distribution_table = pd.DataFrame({
    "Predicted Label": prediction_distribution.index,
    "Count": prediction_distribution.values,
    "Percentage": (
        prediction_distribution.values
        / len(svm_test_predictions)
        * 100
    ).round(2)
})

display(prediction_distribution_table)

,Predicted Label,Count,Percentage
0,0,2488,35.55
1,1,4511,64.45


In [51]:
svm_submission_path = (
    PROJECT_ROOT
    / "submissions"
    / "SVM_Prediction.csv"
)

svm_submission_df = create_submission(
    test_ids=test_df[ID_COLUMN],
    predictions=svm_test_predictions,
    output_path=svm_submission_path,
    id_column=ID_COLUMN,
    label_column=LABEL_COLUMN
)

display(svm_submission_df.head())

print("SVM submission saved to:")
print(svm_submission_path)

,id,label
0,59218,1
1,37110,1
2,23200,0
3,e3357348-166e-4847-a06d-158b7cd83aa5,1
4,61615,1


SVM submission saved to:
c:\Users\akshi\Downloads\TheHomiesML-1\submissions\SVM_Prediction.csv


In [52]:
saved_svm_submission = pd.read_csv(
    svm_submission_path,
    dtype={ID_COLUMN: "string"}
)

expected_test_ids = (
    test_df[ID_COLUMN]
    .astype("string")
    .reset_index(drop=True)
)

assert saved_svm_submission.columns.tolist() == [
    ID_COLUMN,
    LABEL_COLUMN
]

assert len(saved_svm_submission) == len(test_df)

assert (
    saved_svm_submission[ID_COLUMN].tolist()
    == expected_test_ids.tolist()
)

assert saved_svm_submission[LABEL_COLUMN].isnull().sum() == 0

assert set(saved_svm_submission[LABEL_COLUMN].unique()).issubset(
    set(train_df[LABEL_COLUMN].unique())
)

print("Final submission verified successfully.")
print("Rows:", len(saved_svm_submission))
print("Columns:", saved_svm_submission.columns.tolist())
print("Missing predictions: 0")
print("Test ID order preserved.")

Final submission verified successfully.
Rows: 6999
Columns: ['id', 'label']
Missing predictions: 0
Test ID order preserved.
